# 01 — Exploratory Data Analysis

Goal: understand the shape, quality, and distributions of the Olist dataset before any modeling.

Approach:
- Load raw CSVs with pandas
- Summarize shapes, memory, missingness
- Visualize volume over time, order status, price distribution

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import DATA_RAW, CSV_FILES
from src.viz import PALETTE, save

sns.set_style("whitegrid")
pd.set_option('display.max_columns', None)
print("Data dir:", DATA_RAW.resolve())

In [ ]:
dataframes = {}
for name, filename in CSV_FILES.items():
    path = DATA_RAW / filename
    if path.exists():
        dataframes[name] = pd.read_csv(path)
        print(f"OK {name}: {dataframes[name].shape}")
    else:
        print(f"MISS {path}")

In [ ]:
summary = pd.DataFrame({
    name: {
        'rows': len(df),
        'cols': df.shape[1],
        'missing_pct': round(df.isnull().sum().sum() / df.size * 100, 2),
        'memory_mb': round(df.memory_usage(deep=True).sum() / 1e6, 2),
    }
    for name, df in dataframes.items()
}).T
summary

In [ ]:
tables_with_missing = {
    n: df.isnull().mean()
    for n, df in dataframes.items()
    if df.isnull().sum().sum() > 0
}
missing_df = pd.DataFrame(tables_with_missing).T

fig, ax = plt.subplots(figsize=(10, max(4, len(missing_df) * 0.6)))
sns.heatmap(
    missing_df.fillna(0),
    cmap='Reds',
    annot=True,
    fmt='.1%',
    cbar_kws={'label': 'Missing rate'},
    ax=ax,
)
ax.set_title('Missing Value Rate by Table')
save(fig, '01_missing_values')
plt.show()

In [ ]:
orders = dataframes['orders'].copy()
status_counts = orders['order_status'].value_counts()

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x=status_counts.index, y=status_counts.values, hue=status_counts.index, legend=False,
            palette='viridis', ax=ax)
ax.set_title('Order Status Distribution')
ax.set_ylabel('Orders')
for i, v in enumerate(status_counts.values):
    ax.text(i, v + 100, f'{v:,}', ha='center', fontsize=9)
save(fig, '01_order_status')
plt.show()

In [ ]:
orders['order_purchase_timestamp'] = pd.to_datetime(
    orders['order_purchase_timestamp'], errors='coerce'
)
print("Range:", orders['order_purchase_timestamp'].min(), "->",
      orders['order_purchase_timestamp'].max())

monthly = (
    orders.set_index('order_purchase_timestamp')
    .resample('MS').size()
)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly.index, monthly.values, marker='o', color=PALETTE['primary'])
ax.set_title('Monthly Order Volume')
ax.set_ylabel('Orders')
ax.set_xlabel('Month')
save(fig, '01_monthly_volume')
plt.show()

In [ ]:
prices = dataframes['order_items']['price']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(prices, bins=60, color=PALETTE['primary'], edgecolor='white')
axes[0].set_title('Order Item Price')
axes[0].set_xlabel('Price (BRL)')

axes[1].hist(np.log1p(prices), bins=60,
             color=PALETTE['accent'], edgecolor='white')
axes[1].set_title('log(1 + Price)')
axes[1].set_xlabel('log BRL')

save(fig, '01_price_distribution')
plt.show()

prices.describe()

## Findings

- 9 tables; ~120 MB total; ~100K orders across Sept 2016 – Oct 2018
- `orders.order_status` is dominated by `delivered` (~97%)
- `order_items.price` is right-skewed; median ~R$75, long tail to ~R$6,700
- Missing data concentrates in delivery timestamps (undelivered orders) and product metadata
- Two natural analysis windows: full dataset for volume; 2017-01 → 2018-08 for revenue trends (avoids incomplete boundary months)